# Citi Bike NYC — PySpark Data Processing Pipeline
**IDS706 PySpark Assignment**

This notebook builds a distributed data pipeline on **~2.6 GB of Citi Bike NYC trip data (2024)** joined with **NOAA daily weather (Central Park)**, and analyzes how Spark optimizes it.

| Section | Rubric item |
|---|---|
| 0. Setup | – |
| 1. Load data (CSV → Parquet) | Dataset Selection and Loading |
| 2. Transformations (filter, withColumn, join, groupBy) | Transformation |
| 3. SQL queries | SQL query |
| 4. Write results | Output Writing |
| 5. Execution plans & optimization | Optimization / Execution Plan / Performance Analysis |
| 6. Caching | Bonus |
| 7. Actions vs Transformations | Actions vs Transformations |
| 8. MLlib (optional) | MLlib Usage |

> Spark UI: keep the session running and open the URL printed below (usually http://localhost:4040) to take screenshots.

## 0. Setup

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

RAW_TRIPS = "../data/raw/*.csv"
RAW_WEATHER = "../data/weather/nyc_weather_2024.csv"
PARQUET_TRIPS = "../data/parquet/trips"
OUTPUT_DIR = "../output"

spark = (
    SparkSession.builder
    .appName("citibike-pipeline")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    # Default is 200 shuffle partitions; far too many for one laptop -> lots of tiny tasks.
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/19 11:26:23 WARN Utils: Your hostname, zhangqianchundeMacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.4.149 instead (on interface en0)
26/11/19 11:26:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/zhangqianchun/pyspark-citibike/.venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/19 11:26:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.3.0
Spark UI: http://172.20.4.149:4040


In [2]:
def timed(label, fn):
    """Run fn(), print elapsed seconds, return (result, seconds)."""
    t0 = time.time()
    out = fn()
    dt = time.time() - t0
    print(f"{label}: {dt:.2f}s")
    return out, dt

## 1. Dataset Selection and Loading

**Trips:** Citi Bike NYC system data, monthly files from https://s3.amazonaws.com/tripdata/index.html (each month is split into several CSVs, ~2.6 GB of raw CSV total).
**Weather:** NOAA GHCN Daily summaries, station USW00094728 (NY Central Park), 2024 — precipitation, max/min temperature, wind.

We define the schema explicitly instead of using `inferSchema`, which would force Spark to scan all 2.6 GB an extra time just to guess types.

In [3]:
trip_schema = StructType([
    StructField("ride_id", StringType()),
    StructField("rideable_type", StringType()),
    StructField("started_at", StringType()),
    StructField("ended_at", StringType()),
    StructField("start_station_name", StringType()),
    StructField("start_station_id", StringType()),
    StructField("end_station_name", StringType()),
    StructField("end_station_id", StringType()),
    StructField("start_lat", DoubleType()),
    StructField("start_lng", DoubleType()),
    StructField("end_lat", DoubleType()),
    StructField("end_lng", DoubleType()),
    StructField("member_casual", StringType()),
])

trips_csv = (
    spark.read.option("header", True).schema(trip_schema).csv(RAW_TRIPS)
    .withColumn("started_at", F.to_timestamp("started_at"))
    .withColumn("ended_at", F.to_timestamp("ended_at"))
)
csv_rows, csv_count_time = timed("Count rows from raw CSV", trips_csv.count)
print(f"Rows: {csv_rows:,}")
trips_csv.printSchema()
trips_csv.show(3, truncate=False)

26/11/19 11:26:25 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/*.csv.
java.io.FileNotFoundException: File ../data/raw/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:401)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:209)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

Count rows from raw CSV: 2.43s
Rows: 14,110,047
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)

+----------------+-------------+-----------------------+-----------------------+------------------+----------------+-----------------------+--------------+-----------+------------+-----------------+------------------+-------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name|start_station_id|end_station_name       |end_s

### Convert to Parquet, partitioned by month
Parquet is columnar and compressed, so later queries read only the columns they need (**column pruning**) and can skip row groups using min/max statistics (**predicate pushdown**). Partitioning by `ym` (year-month) enables **partition pruning**: a filter on `ym` skips whole folders.

In [4]:
_, _ = timed("Write Parquet", lambda: (
    trips_csv
    .withColumn("ym", F.date_format("started_at", "yyyy-MM"))
    .write.mode("overwrite")
    .partitionBy("ym")
    .parquet(PARQUET_TRIPS)
))

Write Parquet: 17.96s


In [5]:
trips = spark.read.parquet(PARQUET_TRIPS)
pq_rows, pq_count_time = timed("Count rows from Parquet", trips.count)
print(f"Rows: {pq_rows:,}  |  CSV count {csv_count_time:.2f}s vs Parquet count {pq_count_time:.2f}s")
print("Months available:", [r.ym for r in trips.select("ym").distinct().orderBy("ym").collect()])

Count rows from Parquet: 0.18s
Rows: 14,110,047  |  CSV count 2.43s vs Parquet count 0.18s
Months available: ['2024-05', '2024-06', '2024-07', '2024-08']


In [6]:
weather = (
    spark.read.option("header", True).option("inferSchema", True)   # tiny file: inferSchema is fine here
    .csv(RAW_WEATHER)
    .select(
        F.to_date("DATE").alias("date"),
        F.col("PRCP").cast("double").alias("precip_in"),
        F.col("TMAX").cast("double").alias("tmax_f"),
        F.col("TMIN").cast("double").alias("tmin_f"),
        F.col("AWND").cast("double").alias("wind_mph"),
    )
)
weather.show(3)
print("Weather rows:", weather.count())

+----------+---------+------+------+--------+
|      date|precip_in|tmax_f|tmin_f|wind_mph|
+----------+---------+------+------+--------+
|2024-01-01|     0.03|  47.0|  35.0|    3.36|
|2024-01-02|      0.0|  42.0|  29.0|    4.03|
|2024-01-03|      0.0|  43.0|  34.0|    4.92|
+----------+---------+------+------+--------+
only showing top 3 rows
Weather rows: 366


## 2. Transformations

Optimization choices in this section:
1. **Column pruning first** — `select` only the 10 columns we need (drops `ride_id`, station ids).
2. **Filters early** — null/coordinate filters on raw columns come right after the read, so Spark pushes them into the Parquet scan (`PushedFilters`).
3. **Broadcast join** — the weather table has ~366 rows, so it is broadcast to every executor instead of shuffling 2.6 GB of trips.

In [7]:
NYC_BBOX = dict(lat_min=40.4, lat_max=41.0, lng_min=-74.3, lng_max=-73.6)

trips_clean = (
    trips
    # column pruning
    .select("rideable_type", "started_at", "ended_at", "start_station_name", "end_station_name",
            "start_lat", "start_lng", "end_lat", "end_lng", "member_casual", "ym")
    # FILTER 1: drop rows with missing station / coordinates (pushed down to the Parquet scan)
    .filter(F.col("start_station_name").isNotNull() & F.col("end_station_name").isNotNull()
            & F.col("end_lat").isNotNull() & F.col("end_lng").isNotNull())
    # FILTER 2: keep coordinates inside the NYC area (removes GPS glitches)
    .filter(F.col("start_lat").between(NYC_BBOX["lat_min"], NYC_BBOX["lat_max"])
            & F.col("start_lng").between(NYC_BBOX["lng_min"], NYC_BBOX["lng_max"]))
    # withColumn: derived features
    .withColumn("duration_min", (F.unix_timestamp("ended_at") - F.unix_timestamp("started_at")) / 60.0)
    .withColumn("date", F.to_date("started_at"))
    .withColumn("hour", F.hour("started_at"))
    .withColumn("day_of_week", F.date_format("started_at", "E"))
    .withColumn("is_weekend", F.dayofweek("started_at").isin(1, 7))
    # haversine distance in km
    .withColumn("distance_km", 2 * 6371 * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(F.col("end_lat") - F.col("start_lat")) / 2), 2)
        + F.cos(F.radians("start_lat")) * F.cos(F.radians("end_lat"))
        * F.pow(F.sin(F.radians(F.col("end_lng") - F.col("start_lng")) / 2), 2)
    )))
    # FILTER 3: remove implausible trips (< 1 min or > 3 hours)
    .filter(F.col("duration_min").between(1, 180))
)

# JOIN: broadcast the small weather table
trips_enriched = (
    trips_clean
    .join(F.broadcast(weather), on="date", how="left")
    .withColumn("is_rainy", F.col("precip_in") > 0.1)
)
trips_enriched.show(5)

+----------+-------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+-----------------+------------------+-------------+-------+------------------+----+-----------+----------+------------------+---------+------+------+--------+--------+
|      date|rideable_type|          started_at|            ended_at|  start_station_name|    end_station_name|         start_lat|         start_lng|          end_lat|           end_lng|member_casual|     ym|      duration_min|hour|day_of_week|is_weekend|       distance_km|precip_in|tmax_f|tmin_f|wind_mph|is_rainy|
+----------+-------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+-----------------+------------------+-------------+-------+------------------+----+-----------+----------+------------------+---------+------+------+--------+--------+
|2024-06-14| classic_bike|2024-06-14 08:41:...|2024-

In [8]:
clean_rows, _ = timed("Count cleaned + enriched trips", trips_enriched.count)
print(f"Kept {clean_rows:,} of {pq_rows:,} rows ({clean_rows / pq_rows:.1%})")

Count cleaned + enriched trips: 1.86s
Kept 14,046,688 of 14,110,047 rows (99.6%)


### groupBy + aggregations

In [9]:
# Station x hour demand profile
station_hour = (
    trips_enriched
    .groupBy("start_station_name", "hour", "is_weekend")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("duration_min"), 2).alias("avg_duration_min"),
        F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
        F.round(F.avg(F.when(F.col("member_casual") == "member", 1).otherwise(0)), 3).alias("member_share"),
    )
)
station_hour.orderBy(F.desc("trips")).show(10, truncate=False)

# Daily totals with weather (used by SQL + MLlib below)
daily = (
    trips_enriched
    .groupBy("date", "member_casual", "is_weekend", "is_rainy", "precip_in", "tmax_f")
    .agg(F.count("*").alias("trips"), F.round(F.avg("duration_min"), 2).alias("avg_duration_min"))
)
daily.orderBy("date").show(5)

+-----------------------------+----+----------+-----+----------------+---------------+------------+
|start_station_name           |hour|is_weekend|trips|avg_duration_min|avg_distance_km|member_share|
+-----------------------------+----+----------+-----+----------------+---------------+------------+
|E 47 St & Park Ave           |17  |false     |5116 |13.99           |2.5            |0.92        |
|West St & Chambers St        |18  |false     |4740 |15.87           |3.0            |0.765       |
|North Moore St & Greenwich St|17  |false     |4643 |10.41           |1.88           |0.939       |
|9 Ave & W 33 St              |17  |false     |4612 |12.26           |2.12           |0.859       |
|E 40 St & Park Ave           |17  |false     |4449 |13.55           |2.41           |0.864       |
|1 Ave & E 68 St              |17  |false     |4418 |15.36           |2.8            |0.9         |
|West St & Chambers St        |17  |false     |4297 |16.23           |3.14           |0.743       |


+----------+-------------+----------+--------+---------+------+------+----------------+
|      date|member_casual|is_weekend|is_rainy|precip_in|tmax_f| trips|avg_duration_min|
+----------+-------------+----------+--------+---------+------+------+----------------+
|2024-05-31|       member|     false|   false|      0.0|  75.0|   701|           21.54|
|2024-05-31|       casual|     false|   false|      0.0|  75.0|   453|           28.96|
|2024-06-01|       member|      true|   false|      0.0|  83.0|118836|            13.5|
|2024-06-01|       casual|      true|   false|      0.0|  83.0| 53208|           21.44|
|2024-06-02|       casual|      true|   false|      0.0|  81.0| 44461|           21.81|
+----------+-------------+----------+--------+---------+------+------+----------------+
only showing top 5 rows


## 3. SQL Queries
Register the DataFrames as temp views and query them with Spark SQL. SQL and DataFrame code compile to the same Catalyst plan.

In [10]:
trips_enriched.createOrReplaceTempView("trips")
daily.createOrReplaceTempView("daily")

**Query 1 — Top 10 start stations, with member share and average trip length**

In [11]:
q1 = spark.sql("""
    SELECT start_station_name,
           COUNT(*)                                                      AS trips,
           ROUND(AVG(duration_min), 1)                                   AS avg_min,
           ROUND(AVG(CASE WHEN member_casual = 'member' THEN 1 ELSE 0 END), 3) AS member_share
    FROM trips
    GROUP BY start_station_name
    ORDER BY trips DESC
    LIMIT 10
""")
q1.show(truncate=False)

+------------------------+-----+-------+------------+
|start_station_name      |trips|avg_min|member_share|
+------------------------+-----+-------+------------+
|West St & Chambers St   |48608|17.3   |0.674       |
|W 21 St & 6 Ave         |46868|10.4   |0.875       |
|University Pl & E 14 St |46454|11.0   |0.822       |
|Pier 61 at Chelsea Piers|43216|15.1   |0.712       |
|8 Ave & W 31 St         |42191|13.1   |0.807       |
|11 Ave & W 41 St        |42120|12.6   |0.813       |
|Broadway & W 58 St      |41551|18.4   |0.69        |
|Lafayette St & E 8 St   |41471|11.3   |0.821       |
|12 Ave & W 40 St        |39188|19.6   |0.596       |
|Broadway & E 21 St      |38393|11.7   |0.834       |
+------------------------+-----+-------+------------+



**Query 2 — Does rain reduce ridership? Average daily trips on rainy vs dry days, members vs casual riders**

In [12]:
q2 = spark.sql("""
    SELECT member_casual,
           CASE WHEN is_rainy THEN 'rainy (>0.1 in)' ELSE 'dry' END AS weather,
           COUNT(DISTINCT date)              AS days,
           ROUND(AVG(trips), 0)              AS avg_daily_trips,
           ROUND(AVG(avg_duration_min), 2)   AS avg_duration_min
    FROM daily
    GROUP BY member_casual, is_rainy
    ORDER BY member_casual, weather
""")
q2.show(truncate=False)

+-------------+---------------+----+---------------+----------------+
|member_casual|weather        |days|avg_daily_trips|avg_duration_min|
+-------------+---------------+----+---------------+----------------+
|casual       |dry            |73  |36223.0        |19.16           |
|casual       |rainy (>0.1 in)|20  |29562.0        |17.89           |
|member       |dry            |73  |119278.0       |12.07           |
|member       |rainy (>0.1 in)|20  |105194.0       |11.4            |
+-------------+---------------+----+---------------+----------------+



**Query 3 — Peak hours: rank each hour within weekday vs weekend (window function)**

In [13]:
q3 = spark.sql("""
    WITH hourly AS (
        SELECT is_weekend, hour, COUNT(*) AS trips
        FROM trips
        GROUP BY is_weekend, hour
    )
    SELECT CASE WHEN is_weekend THEN 'weekend' ELSE 'weekday' END AS day_type,
           hour, trips,
           RANK() OVER (PARTITION BY is_weekend ORDER BY trips DESC) AS rank_in_day_type
    FROM hourly
    QUALIFY rank_in_day_type <= 3
    ORDER BY day_type, rank_in_day_type
""")
q3.show()

+--------+----+------+----------------+
|day_type|hour| trips|rank_in_day_type|
+--------+----+------+----------------+
| weekday|  17|998432|               1|
| weekday|  18|947298|               2|
| weekday|  19|743550|               3|
| weekend|  14|305502|               1|
| weekend|  15|304524|               2|
| weekend|  13|298212|               3|
+--------+----+------+----------------+



## 4. Write Results
- `station_hour` → Parquet partitioned by `is_weekend` (downstream readers can prune).
- Small result tables → `coalesce(1)` so they produce one file instead of 16 tiny ones.

In [14]:
_, _ = timed("Write station_hour", lambda: station_hour.write.mode("overwrite")
             .partitionBy("is_weekend").parquet(f"{OUTPUT_DIR}/station_hour"))
_, _ = timed("Write daily", lambda: daily.coalesce(1).write.mode("overwrite").parquet(f"{OUTPUT_DIR}/daily"))
_, _ = timed("Write SQL results", lambda: [
    q.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUTPUT_DIR}/{name}")
    for name, q in [("top_stations", q1), ("rain_effect", q2), ("peak_hours", q3)]
])
print("Read back:", spark.read.parquet(f"{OUTPUT_DIR}/station_hour").count(), "station-hour rows")

Write station_hour: 2.95s


Write daily: 2.32s


Write SQL results: 7.12s
Read back: 101403 station-hour rows


## 5. Execution Plans & Optimization

### 5.1 Physical plan of the optimized pipeline
Look for:
- `FileScan parquet ... PushedFilters: [IsNotNull(start_station_name), ...]` → filters pushed into the scan
- `ReadSchema:` → only the selected columns are read (column pruning)
- `BroadcastHashJoin ... BuildRight` → no shuffle of the big trips table for the join
- `Exchange hashpartitioning(...)` → the shuffle required by `groupBy` (the main remaining cost)

In [15]:
station_hour.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- HashAggregate (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Project (9)
            +- BroadcastHashJoin LeftOuter BuildRight (8)
               :- Project (3)
               :  +- Filter (2)
               :     +- Scan parquet  (1)
               +- BroadcastExchange (7)
                  +- Project (6)
                     +- Filter (5)
                        +- Scan csv  (4)


(1) Scan parquet 
Output [10]: [started_at#103, ended_at#104, start_station_name#105, end_station_name#107, start_lat#109, start_lng#110, end_lat#111, end_lng#112, member_casual#113, ym#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/zhangqianchun/pyspark-citibike/data/parquet/trips]
PushedFilters: [IsNotNull(start_lat), IsNotNull(start_lng), IsNotNull(start_station_name), IsNotNull(end_station_name), IsNotNull(end_lat), IsNotNull(end_lng), GreaterThanOrEqual(start_lat,40.4), LessThanOrEqual(start_lat,41.0), GreaterThanOrEqual(start

### 5.2 Partition pruning
Filtering on the partition column `ym` means Spark only lists/reads that month's folder — see `PartitionFilters` in the plan.

In [16]:
one_month = sorted(r.ym for r in trips.select("ym").distinct().collect())[0]
pruned = trips.filter(F.col("ym") == one_month).groupBy("member_casual").count()
pruned.explain(mode="formatted")
_, _ = timed(f"Count trips in {one_month} only (partition pruning)", pruned.collect)

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [member_casual#113, ym#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/zhangqianchun/pyspark-citibike/data/parquet/trips]
PartitionFilters: [isnotnull(ym#114), (ym#114 = 2024-05)]
ReadSchema: struct<member_casual:string>

(2) Project
Output [1]: [member_casual#113]
Input [2]: [member_casual#113, ym#114]

(3) HashAggregate
Input [1]: [member_casual#113]
Keys [1]: [member_casual#113]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#952L]
Results [2]: [member_casual#113, count#953L]

(4) Exchange
Input [2]: [member_casual#113, count#953L]
Arguments: hashpartitioning(member_casual#113, 16), ENSURE_REQUIREMENTS, [plan_id=3422]

(5) HashAggregate
Input [2]: [member_casual#113, count#953L]
Keys [1]: [member_casual#113]
Functions [1]: [count(1)]
Aggregate Attributes [1]

### 5.3 Naive vs optimized pipeline
The **naive** version reads raw CSV, joins *before* filtering, and disables broadcast joins (forcing a `SortMergeJoin` with a shuffle on both sides). The **optimized** version is the pipeline above (Parquet + early filters + column pruning + broadcast join). Both compute the same station-hour aggregation.

In [17]:
def naive_pipeline():
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # force shuffle join
    df = (
        trips_csv                                                     # raw CSV, all columns
        .withColumn("date", F.to_date("started_at"))
        .join(weather, on="date", how="left")                         # join BEFORE filtering
        .withColumn("duration_min", (F.unix_timestamp("ended_at") - F.unix_timestamp("started_at")) / 60.0)
        .withColumn("hour", F.hour("started_at"))
        .withColumn("is_weekend", F.dayofweek("started_at").isin(1, 7))
        .filter(F.col("start_station_name").isNotNull() & F.col("end_station_name").isNotNull())
        .filter(F.col("duration_min").between(1, 180))
        .groupBy("start_station_name", "hour", "is_weekend").count()
    )
    return df

naive = naive_pipeline()
naive.explain(mode="formatted")
_, naive_time = timed("NAIVE pipeline", naive.collect)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # restore default (10MB)

== Physical Plan ==
AdaptiveSparkPlan (17)
+- HashAggregate (16)
   +- Exchange (15)
      +- HashAggregate (14)
         +- Project (13)
            +- SortMergeJoin LeftOuter (12)
               :- Sort (6)
               :  +- Exchange (5)
               :     +- Project (4)
               :        +- Project (3)
               :           +- Filter (2)
               :              +- Scan csv  (1)
               +- Sort (11)
                  +- Exchange (10)
                     +- Project (9)
                        +- Filter (8)
                           +- Scan csv  (7)


(1) Scan csv 
Output [4]: [started_at#2, ended_at#3, start_station_name#4, end_station_name#6]
Batched: false
Location: InMemoryFileIndex [file:/Users/zhangqianchun/pyspark-citibike/data/raw/202406-citibike-tripdata_1.csv, ... 14 entries]
PushedFilters: [IsNotNull(start_station_name), IsNotNull(end_station_name)]
ReadSchema: struct<started_at:string,ended_at:string,start_station_name:string,end_station_name:

NAIVE pipeline: 14.91s


In [18]:
optimized = station_hour.select("start_station_name", "hour", "is_weekend", "trips")
_, opt_time = timed("OPTIMIZED pipeline", optimized.collect)
print(f"Speed-up: {naive_time / opt_time:.1f}x")

OPTIMIZED pipeline: 2.02s
Speed-up: 7.4x


## 6. Caching (bonus)
`trips_enriched` is reused by several queries. Without caching, **every action re-reads Parquet and re-runs all filters, withColumns and the join**. After `.cache()`, the first action materializes it in memory and later actions read from memory.

In [19]:
trips_enriched.unpersist()
_, nc1 = timed("No cache  - action 1 (count)", trips_enriched.count)
_, nc2 = timed("No cache  - action 2 (avg duration by type)",
               lambda: trips_enriched.groupBy("rideable_type").avg("duration_min").collect())

trips_enriched.cache()
_, c0 = timed("Cache     - materialize (count)", trips_enriched.count)
_, c1 = timed("Cache     - action 1 (count)", trips_enriched.count)
_, c2 = timed("Cache     - action 2 (avg duration by type)",
              lambda: trips_enriched.groupBy("rideable_type").avg("duration_min").collect())

print(f"\nRepeated count: {nc1:.2f}s -> {c1:.2f}s  ({nc1 / max(c1, 1e-3):.1f}x faster)")
print(f"Repeated aggregation: {nc2:.2f}s -> {c2:.2f}s  ({nc2 / max(c2, 1e-3):.1f}x faster)")

No cache  - action 1 (count): 1.24s


No cache  - action 2 (avg duration by type): 1.16s


Cache     - materialize (count): 6.97s
Cache     - action 1 (count): 0.04s
Cache     - action 2 (avg duration by type): 0.56s

Repeated count: 1.24s -> 0.04s  (28.1x faster)
Repeated aggregation: 1.16s -> 0.56s  (2.1x faster)


In [20]:
# The plan now starts from InMemoryTableScan instead of FileScan parquet.
# Spark UI -> Storage tab shows the cached DataFrame and its size in memory.
trips_enriched.groupBy("rideable_type").count().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[rideable_type#102], functions=[count(1)])
   +- Exchange hashpartitioning(rideable_type#102, 16), ENSURE_REQUIREMENTS, [plan_id=4356]
      +- HashAggregate(keys=[rideable_type#102], functions=[partial_count(1)])
         +- InMemoryTableScan [rideable_type#102]
               +- InMemoryRelation [date#195, rideable_type#102, started_at#103, ended_at#104, start_station_name#105, end_station_name#107, start_lat#109, start_lng#110, end_lat#111, end_lng#112, member_casual#113, ym#114, duration_min#194, hour#196, day_of_week#197, is_weekend#198, distance_km#199, precip_in#159, tmax_f#160, tmin_f#161, wind_mph#162, is_rainy#200], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- AdaptiveSparkPlan isFinalPlan=true
                        +- == Final Plan ==
                           ResultQueryStage 1
                           +- *(2) Project [date#195, rideable_type#102, started_at#10

## 7. Actions vs Transformations
**Transformations** (`filter`, `withColumn`, `select`, `join`, `groupBy`) are **lazy**: they only add steps to a logical plan, so they return instantly and launch **no Spark jobs**.
**Actions** (`count`, `show`, `collect`, `write`) are **eager**: they make Spark optimize the plan and actually execute it.

We prove this by counting how many Spark jobs each step launches.

In [21]:
sc = spark.sparkContext
fresh = spark.read.parquet(PARQUET_TRIPS)   # fresh DataFrame, not cached

sc.setJobGroup("transformations", "lazy demo")
t0 = time.time()
lazy_df = (
    fresh
    .filter(F.col("member_casual") == "casual")
    .withColumn("duration_min", (F.unix_timestamp("ended_at") - F.unix_timestamp("started_at")) / 60.0)
    .filter(F.col("duration_min") > 30)
    .select("start_station_name", "duration_min")
    .groupBy("start_station_name").agg(F.avg("duration_min").alias("avg_min"))
)
t_transform = time.time() - t0
jobs_transform = len(sc.statusTracker().getJobIdsForGroup("transformations"))

sc.setJobGroup("action", "eager demo")
t0 = time.time()
n_stations = lazy_df.count()
t_action = time.time() - t0
jobs_action = len(sc.statusTracker().getJobIdsForGroup("action"))
sc.setJobGroup(None, None)

print(f"Transformations: {t_transform*1000:.1f} ms, Spark jobs launched = {jobs_transform}")
print(f"Action (count):  {t_action*1000:.1f} ms, Spark jobs launched = {jobs_action}")
print(f"Result: {n_stations:,} stations with casual rides > 30 min")

Transformations: 27.8 ms, Spark jobs launched = 0
Action (count):  762.8 ms, Spark jobs launched = 3
Result: 2,157 stations with casual rides > 30 min


In [22]:
# A transformation only returns a new DataFrame (a plan), not data:
print(type(lazy_df))
lazy_df.explain(mode="simple")   # the plan exists, but nothing has run yet until an action is called

<class 'pyspark.sql.classic.dataframe.DataFrame'>
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[start_station_name#3262], functions=[avg(duration_min#3273)])
   +- Exchange hashpartitioning(start_station_name#3262, 16), ENSURE_REQUIREMENTS, [plan_id=4485]
      +- HashAggregate(keys=[start_station_name#3262], functions=[partial_avg(duration_min#3273)])
         +- Project [start_station_name#3262, (cast((unix_timestamp(ended_at#3261, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true) - unix_timestamp(started_at#3260, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true)) as double) / 60.0) AS duration_min#3273]
            +- Filter ((isnotnull(member_casual#3270) AND (member_casual#3270 = casual)) AND ((cast((unix_timestamp(ended_at#3261, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true) - unix_timestamp(started_at#3260, yyyy-MM-dd HH:mm:ss, Some(America/New_York), true)) as double) / 60.0) > 30.0))
               +- FileScan parquet [started_at#3260,end

## 8. MLlib (optional): predict hourly ridership
Aggregate to one row per (date, hour), then train a Random Forest regressor on hour, weekend flag and weather.

In [23]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

hourly = (
    trips_enriched
    .groupBy("date", "hour", "is_weekend", "precip_in", "tmax_f", "wind_mph")
    .agg(F.count("*").alias("trips"))
    .withColumn("is_weekend_i", F.col("is_weekend").cast("int"))
    .na.drop()
)
train, test = hourly.randomSplit([0.8, 0.2], seed=42)

ml_pipe = Pipeline(stages=[
    VectorAssembler(inputCols=["hour", "is_weekend_i", "precip_in", "tmax_f", "wind_mph"], outputCol="features"),
    RandomForestRegressor(featuresCol="features", labelCol="trips", numTrees=50, maxDepth=8, seed=42),
])
model = ml_pipe.fit(train)
pred = model.transform(test)

ev = RegressionEvaluator(labelCol="trips", predictionCol="prediction")
print(f"Test RMSE: {ev.evaluate(pred, {ev.metricName: 'rmse'}):,.0f} trips/hour")
print(f"Test R^2:  {ev.evaluate(pred, {ev.metricName: 'r2'}):.3f}")
rf = model.stages[-1]
print("Feature importances:", dict(zip(["hour", "is_weekend", "precip_in", "tmax_f", "wind_mph"],
                                       [round(float(x), 3) for x in rf.featureImportances])))

26/11/19 11:27:41 WARN DAGScheduler: Broadcasting large task binary with size 1159.6 KiB


Test RMSE: 1,780 trips/hour
Test R^2:  0.832
Feature importances: {'hour': 0.929, 'is_weekend': 0.014, 'precip_in': 0.025, 'tmax_f': 0.014, 'wind_mph': 0.018}


## 9. Wrap-up
Before stopping Spark, take the Spark UI screenshots (Jobs, Stages, SQL / DataFrame → query details, Storage).

In [24]:
# spark.stop()   # uncomment AFTER taking Spark UI screenshots